<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_26_practicum_dp_greedy/re_lesson_26_greedy_regex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔎 Урок 26 (додатково) — пошук у тексті: модуль `re` і «жадібність»

Слово «жадібний» у курсі трапляється двічі: **жадібний алгоритм** (урок 26) і **жадібний квантифікатор** у регулярних виразах. Назва спільна, поведінка — різна. Цей ноутбук веде від простого пошуку в рядку до регулярних виразів на журналі водіїв «Смачно + Таксі».

| Розділ | Що всередині |
|---|---|
| 1. Методи рядків | коли `in`, `find`, `replace` досить |
| 2. Перший шаблон | `re.search`, об'єкт `Match`, сирі рядки `r"..."` |
| 3. Будівельні блоки | класи символів, квантифікатори, групи |
| 4. Функції `re` | `search`, `match`, `fullmatch`, `findall`, `finditer`, `sub` |
| 5. Жадібні й нежадібні | `.*` проти `.*?`, backtracking; чим це не жадібний алгоритм |
| 6. Межі regex | чому не для HTML |
| 7. Вправи | 5 задач на журналі водіїв |

Базова таблиця функцій `re` уже була в уроці 12 (`notes_modules.ipynb`, частина 11). Тут — детальніше й з пастками.

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам.

---
## 1. Спершу — методи рядків

Якщо шукаєш **конкретний фіксований текст**, регулярні вирази не потрібні: методи рядків простіші, читабельніші й швидші.

In [ ]:
text = "Курс Python розробки"

print("Python" in text)                    # чи є
print(text.find("Python"))                 # де (або -1)
print(text.replace("Python", "Python 3"))  # замінити
print(text.startswith("Курс"))

Регулярні вирази потрібні, коли текст **змінюється за шаблоном**: «будь-яке число після `#`», «дві цифри, двокрапка, дві цифри», «усі пробіли підряд». Методам рядків такого не пояснити.

---
## 2. Перший шаблон: `re.search`

`re.search(шаблон, текст)` шукає **перший** збіг будь-де в тексті. Повертає об'єкт `Match` або `None`.

`\d` означає «будь-яка цифра», `{2}` — «рівно двічі»:

In [ ]:
import re

m = re.search(r"(\d{2}):(\d{2})", "Старт зміни о 09:15")
print(m)
print(m.group())      # увесь збіг
print(m.group(1))     # перша група в дужках
print(m.groups())     # усі групи
print(m.start())      # позиція початку

### Сирі рядки `r"..."`

У звичайному рядку Python сам обробляє `\`: `"\n"` — перенос, `"\b"` — службовий символ backspace. А в regex `\b` означає **межу слова**. Щоб Python не чіпав зворотні скісні, шаблони пишуть **сирими рядками** з префіксом `r`.

**🔮 Прогноз:** що виведуть обидва рядки?

```python
print(re.search("\bOrder", "Order-771"))
print(re.search(r"\bOrder", "Order-771"))
```

<details>
<summary>Відповідь</summary>

`None`, потім `<re.Match object; span=(0, 5), match='Order'>`. Без `r` Python перетворив `"\b"` на символ backspace (`'\x08'`), і regex шукав його — а його в тексті немає.

</details>

In [ ]:
print(re.search("\bOrder", "Order-771"))
print(re.search(r"\bOrder", "Order-771"))
print(repr("\b"))

**Правило:** шаблон regex — завжди `r"..."`.

---
## 3. Будівельні блоки

| Шаблон | Значення | Приклад |
|---|---|---|
| `\d` / `\D` | цифра / не цифра | `\d+` — `771` |
| `\w` / `\W` | літера, цифра або `_` / інше | `\w+` — `Хрещатик` |
| `\s` / `\S` | пробільний символ (пробіл, `\t`, `\n`) / інше | `\s+` |
| `.` | будь-який символ, крім переносу рядка | |
| `[...]` | один символ з набору | `[АБВ]`, `[0-9]` |
| `^` / `$` | початок / кінець рядка | |
| `\b` | межа слова | `\bOrder` |

**Квантифікатори** — скільки разів повторити попереднє:

| Квантифікатор | Скільки |
|---|---|
| `*` | 0 або більше |
| `+` | 1 або більше |
| `?` | 0 або 1 (необов'язкове) |
| `{n}` / `{n,m}` | рівно `n` / від `n` до `m` |

**Групи:** `(...)` — запам'ятати частину збігу; `(?:...)` — лише згрупувати, не запам'ятовуючи; `a|b` — «або».

In [ ]:
text = "Замовлення #10452 підтверджено. Попередні: #892 та Order-771."

print(re.findall(r"(?:#|Order-)\d+", text))   # група без запам'ятовування
print(re.findall(r"(#|Order-)\d+", text))     # звичайна група

Якщо в шаблоні є **звичайна** група, `findall` повертає вміст групи, а не весь збіг — звідси `['#', '#', 'Order-']`. Коли дужки потрібні лише для `|`, пиши `(?:...)`.

---
## 4. Функції модуля `re`

| Функція | Що робить | Повертає |
|---|---|---|
| `re.search(p, s)` | перший збіг будь-де | `Match` або `None` |
| `re.match(p, s)` | збіг лише **на початку** рядка | `Match` або `None` |
| `re.fullmatch(p, s)` | **весь** рядок відповідає шаблону | `Match` або `None` |
| `re.findall(p, s)` | усі збіги | список рядків (або кортежів, якщо груп кілька) |
| `re.finditer(p, s)` | усі збіги по одному | ітератор `Match` (урок 10) |
| `re.sub(p, repl, s)` | замінити всі збіги | новий рядок |

In [ ]:
print(re.match(r"\d+", "Order-771 на 250"))    # на початку — літери
print(re.search(r"\d+", "Order-771 на 250"))   # будь-де
print(re.fullmatch(r"\d{2}:\d{2}", "09:15"))
print(re.fullmatch(r"\d{2}:\d{2}", "09:15 ранку"))

Журнал водія — по рядку на замовлення. `re.M` (multiline) змушує `^` і `$` працювати для **кожного рядка**, а не лише для всього тексту:

In [ ]:
log = """09:15 #104 Хрещатик 250 грн
10:02 #105 Поділ 180 грн
10:40 Order-106 Оболонь 320 грн"""

print(re.findall(r"^(\d{2}:\d{2}) (?:#|Order-)(\d+) .*? (\d+) грн$", log, re.M))

for m in re.finditer(r"(\d+) грн", log):
    print(m.group(1), m.span())

print(re.sub(r"(\d{2})\.(\d{2})\.(\d{4})", r"\3-\2-\1", "15.04.2024"))   # \1 — вміст групи 1

---
## 5. Жадібні й нежадібні квантифікатори

**🔮 Прогноз:** що виведе кожен рядок?

```python
html = "<b>Python</b><i>курс</i>"
print(re.findall(r"<.*>", html))
print(re.findall(r"<.*?>", html))
```

<details>
<summary>Відповідь</summary>

`['<b>Python</b><i>курс</i>']` і `['<b>', '</b>', '<i>', '</i>']`.

</details>

In [ ]:
html = "<b>Python</b><i>курс</i>"
print(re.findall(r"<.*>", html))
print(re.findall(r"<.*?>", html))

**Жадібний** `.*` (за замовчуванням `*` і `+` жадібні):

1. `<` збігається з першою дужкою;
2. `.*` з'їдає **весь** залишок рядка;
3. шаблону ще потрібен `>`, а текст скінчився — рушій **відступає** (backtracking) на один символ назад, потім ще на один… доки не натрапить на `>`. Перший знайдений з кінця `>` — останній у рядку.

**Нежадібний** `.*?` (знак `?` після квантифікатора): бере **якомога менше** і після кожного символу перевіряє, чи не можна вже закінчити. Перший же `>` закриває збіг.

Те саме з лапками в коментарі до замовлення:

In [ ]:
comment = 'Коментар: "без цибулі", "дзвонити в двері"'
print(re.findall(r'"(.*)"', comment))
print(re.findall(r'"(.*?)"', comment))

### Жадібний квантифікатор ≠ жадібний алгоритм

| | Жадібний квантифікатор `.*` | Жадібний алгоритм (урок 26) |
|---|---|---|
| Що «жадібно» | захоплює якомога більше символів | бере найкращий варіант на цьому кроці |
| Чи переглядає рішення | **так**: відступає (backtracking), якщо решта шаблону не збігається | **ніколи**: прийняте рішення остаточне |
| Гарантія | знайде збіг, якщо він існує | оптимальність лише для задач з доведенням |

Спільна лише ідея «спершу — якомога більше». І ще: `.*?` не обов'язково швидший за `.*` — нежадібний перевіряє можливість зупинки після кожного символу, і на деяких текстах це більше роботи.

---
## 6. Межі regex: чому не для HTML

In [ ]:
html = "<div><div>вкладений</div> текст</div>"
print(re.findall(r"<div>(.*?)</div>", html))
print(re.findall(r"<div>(.*)</div>", html))

Жоден варіант не дав «вміст зовнішнього `div`» правильно: нежадібний зупинився на першому `</div>` (внутрішньому), жадібний — захопив теги разом з вмістом. Regex не рахує вкладеність: для нього немає «цей `</div>` закриває той `<div>`».

Реальний HTML ще й буває незакритим, з атрибутами в довільному порядку, коментарями та скриптами. Для HTML використовують парсери, що будують дерево елементів: `html.parser` зі стандартної бібліотеки, BeautifulSoup, lxml. Regex — для простих **плоских** шаблонів: номерів, дат, часу, кодів.

---
# 7. 🛠 Вправи

Журнал диспетчерської за день:

In [ ]:
journal = """Замовлення #10452 підтверджено. Попередні: #892 та Order-771.
Зустрічі: 15.04.2024, 31.02.2024 (помилка) та 09.05.2024.
Коментар  водія:\tклієнт   просив   "без цибулі"  і "дзвонити в двері"."""
print(journal)

### Вправа 1. Номери замовлень

Поверни список усіх номерів виду `#12345` або `Order-999` (разом з `#` / `Order-`).

In [ ]:
def order_numbers(text):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return re.findall(r"(?:#|Order-)\d+", text)
    # END SOLUTION


print(order_numbers(journal))
assert order_numbers(journal) == ["#10452", "#892", "Order-771"]
assert order_numbers("без номерів") == []
print("✅ Вправа 1 пройдена")

### Вправа 2. Дати: формат і календар

Regex перевіряє **формат** `DD.MM.YYYY`, але не **календар**: `31.02.2024` має правильний вигляд, хоча такої дати немає. Знайди всі дати за шаблоном, а потім залиш лише справжні, перевіривши кожну через `datetime.strptime(рядок, "%d.%m.%Y")` (він кидає `ValueError` для неіснуючої дати — урок 13).

Поверни список об'єктів `date` (`.date()` від результату `strptime`).

In [ ]:
from datetime import date, datetime


def valid_dates(text):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    result = []
    for raw in re.findall(r"\b\d{2}\.\d{2}\.\d{4}\b", text):
        try:
            result.append(datetime.strptime(raw, "%d.%m.%Y").date())
        except ValueError:
            print("Календарно неіснуюча дата:", raw)
    return result
    # END SOLUTION


print(valid_dates(journal))
assert valid_dates(journal) == [date(2024, 4, 15), date(2024, 5, 9)]
assert valid_dates("29.02.2024 і 29.02.2023") == [date(2024, 2, 29)]
print("✅ Вправа 2 пройдена")

### Вправа 3. Прибрати зайві пробіли

Заміни кожну послідовність пробільних символів (пробіли, табуляції, переноси) одним пробілом і прибери пробіли по краях.

In [ ]:
def squeeze_spaces(text):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return re.sub(r"\s+", " ", text).strip()
    # END SOLUTION


print(repr(squeeze_spaces("Python   і   модуль\tre  --  це \t \t потужно! ")))
assert squeeze_spaces("Python   і   модуль\tre  --  це \t \t потужно! ") == "Python і модуль re -- це потужно!"
assert squeeze_spaces("  а\n\nб  ") == "а б"
print("✅ Вправа 3 пройдена")

### Вправа 4. Побажання в лапках

Поверни список текстів, що стоять у подвійних лапках, **без самих лапок**. Подумай, який квантифікатор потрібен.

In [ ]:
def quoted(text):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return re.findall(r'"(.*?)"', text)
    # END SOLUTION


print(quoted(journal))
assert quoted(journal) == ["без цибулі", "дзвонити в двері"]
assert quoted('лише "одне"') == ["одне"]
print("✅ Вправа 4 пройдена")

### Вправа 5. Рядки журналу → кортежі

Кожен рядок журналу поїздок має вигляд `ГГ:ХХ #номер район сума грн` (номер може бути і `Order-номер`). Поверни список кортежів `(час, номер, сума)`, де номер і сума — **цілі числа**. Рядки, що не відповідають формату, пропусти.

In [ ]:
trips = """09:15 #104 Хрещатик 250 грн
10:02 #105 Поділ 180 грн
збій зв'язку
10:40 Order-106 Оболонь 320 грн"""


def parse_trips(text):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    pattern = r"^(\d{2}:\d{2}) (?:#|Order-)(\d+) .*? (\d+) грн$"
    return [(t, int(n), int(s)) for t, n, s in re.findall(pattern, text, re.M)]
    # END SOLUTION


print(parse_trips(trips))
assert parse_trips(trips) == [("09:15", 104, 250), ("10:02", 105, 180), ("10:40", 106, 320)]
assert sum(amount for _, _, amount in parse_trips(trips)) == 750
print("✅ Вправа 5 пройдена")

---
## Типові помилки

1. **Шаблон без `r`** — `"\b"` стає backspace, збігу немає (розділ 2).
2. **`re.match` замість `re.search`** — `match` шукає лише з початку рядка.
3. **Звичайна група там, де потрібен весь збіг** — `findall` повертає вміст групи; для `|` пиши `(?:...)`.
4. **Жадібний `.*` між розділювачами** — захоплює все до останнього розділювача; потрібен `.*?` або точніший клас (`[^"]*`).
5. **Regex для перевірки змісту** — формат дати ≠ справжня дата; перевіряй через `datetime`.
6. **Regex для HTML** — використовуй парсер.

## Далі

- Урок 26 — жадібні алгоритми й динамічне програмування: [книга](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_26/).
- Документація: [`re`](https://docs.python.org/3/library/re.html), [Regular Expression HOWTO](https://docs.python.org/3/howto/regex.html) (розділ «Greedy versus Non-Greedy»), [`datetime.strptime`](https://docs.python.org/3/library/datetime.html#datetime.datetime.strptime), [`html.parser`](https://docs.python.org/3/library/html.parser.html).